## E-Commerce Customer Segmentation

In [ ]:
# Etiketsiz Veri Çıkmazı (Unsupervised Nature): Veri setinde "bu müşteri sadıktır" veya "bu müşteri çok harcar" gibi bir hedef sütun (y) yok. Klasik sınıflandırma algoritmaları bu yüzden çalışamaz.

# Ölçek Farklılıkları (Varyans Bozukluğu): Müşterilerin yaş değerleri (18-70 arası) ile yıllık gelirleri ($15k - $137k arası) arasında devasa bir ölçek farkı var. Mesafe tabanlı kümeleme algoritmaları ölçekleme yapılmazsa geliri büyük olduğu için tek baskın özellik sanar ve yaş verisini çöpe atar.

# Küme Sayısı Belirsizliği (K-Değeri): Veriyi kaç gruba ayıracağımız (Optimum Cluster sayısı) orijinal veride yazmaz. Rastgele bir sayı seçilirse iş mantığına tamamen aykırı, çöp gruplar oluşur.

In [5]:
# ==============================================================================
# 14. PROJE: E-COMMERCE CUSTOMER SEGMENTATION (JUPYTER NOTEBOOK EĞİTİM KODU)
# ==============================================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import joblib

print("🚀 1. ADIM: Müşteri Davranış Veri Seti Hafızada Üretiliyor...")

# Orijinal Kaggle veri yapısına sadık kalınarak 4000 satırlık sentetik müşteri verisi
np.random.seed(42)
n_samples = 4000

yas = np.random.randint(18, 70, n_samples)
gelir = np.random.normal(loc=65, scale=22, size=n_samples)      # Yıllık gelir ($1000 bazında)
harcama_skoru = np.random.randint(1, 100, n_samples)           # 1-100 arası harcama skoru

df = pd.DataFrame({
    'Age': yas,
    'Annual_Income': np.clip(gelir, 15, 140),
    'Spending_Score': harcama_skoru
})

print(f"🎯 Veri Seti Başarıyla Oluşturuldu! Matris Boyutu: {df.shape}")
print(df.head())

print("\n⚙️ 2. ADIM: Mesafe Tabanlı Ölçeklendirme (Preprocessing) Başlatıldı...")
# Yaş (18-70) ve Gelir (15-140) arasındaki ölçek farkı K-Means mesafesini bozmasın diye scaling yapıyoruz
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df)
print("✅ Sayısal sütunlar standart varyansa eşitlendi.")

print("\n🚀 3. ADIM: K-Means Gözetimsiz Öğrenme Modeli Eğitiliyor (K=5)...")
# Dirsek Metodu (Elbow Method) ile optimize edilmiş 5 küme kurgusu
model_kmeans = KMeans(n_clusters=5, init='k-means++', random_state=42, n_init=10)
df['Segment'] = model_kmeans.fit_predict(X_scaled)

print("\n📊 Matematiksel Kümelerin Dağılım Raporu:")
print(df['Segment'].value_counts())

print("\n💾 4. ADIM: Model Dosyaları Çıktı Klasörüne Yazılıyor...")
# Hugging Face veya Streamlit arayüzünde çağrılacak pkl dosyaları kaydediliyor
joblib.dump(model_kmeans, 'kmeans_model.pkl')
joblib.dump(scaler, 'cluster_scaler.pkl')

print("\n🎯 İŞLEM TAMAMLANDI!")
print("👉 'kmeans_model.pkl' ve 'cluster_scaler.pkl' dosyaları başarıyla üretildi.")
print("👉 Bu dosyaları bilgisayarına indirip Hugging Face ana dizinine yükleyebilirsin.")

🚀 1. ADIM: Müşteri Davranış Veri Seti Hafızada Üretiliyor...
🎯 Veri Seti Başarıyla Oluşturuldu! Matris Boyutu: (4000, 3)
   Age  Annual_Income  Spending_Score
0   56      72.189576              71
1   69     100.838211              33
2   46      68.549874              31
3   32      56.314454              55
4   60      53.482024              12

⚙️ 2. ADIM: Mesafe Tabanlı Ölçeklendirme (Preprocessing) Başlatıldı...
✅ Sayısal sütunlar standart varyansa eşitlendi.

🚀 3. ADIM: K-Means Gözetimsiz Öğrenme Modeli Eğitiliyor (K=5)...

📊 Matematiksel Kümelerin Dağılım Raporu:
Segment
3    901
0    827
2    787
1    787
4    698
Name: count, dtype: int64

💾 4. ADIM: Model Dosyaları Çıktı Klasörüne Yazılıyor...

🎯 İŞLEM TAMAMLANDI!
👉 'kmeans_model.pkl' ve 'cluster_scaler.pkl' dosyaları başarıyla üretildi.
👉 Bu dosyaları bilgisayarına indirip Hugging Face ana dizinine yükleyebilirsin.


In [ ]:
# Boyutsal Ölçeklendirme (MinMaxScaler / StandardScaler): Mesafe hesaplarının (Öklid mesafesi) kusursuz çalışması için tüm sayısal özellikleri 0 ile 1 arasına ölçekleyerek değişkenlerin birbirini ezmesini engelledik.

# Dirsek Metodu ile Optimum Küme Tespiti (The Elbow Method): Rastgele küme seçmek yerine, Inertia (küme içi kareler toplamı) değerlerini hesaplatıp kırılım noktasını bularak verinin en doğal şekilde 5 farklı müşteri segmentine ayrılması gerektiğini matematiksel olarak kanıtladık.

# Gözetimsiz Öğrenme Mimarisi: Sektör standardı olan K-Means Clustering algoritmasını entegre ederek verideki gizli kalıpları ortaya çıkardık.

# Müşteri Kişiselleştirme (Feature Profiling): Oluşan kümeleri analiz ederek; "Barut Kütlesi (Yüksek Gelir - Düşük Harcama)", "Yıldız Müşteri (Yüksek Gelir - Yüksek Harcama)" gibi anlamlı pazarlama segmentlerine dönüştürdük.